# Backpropagation Implementation Notes

## Forward Pass
- Store intermediates: keep every pre-activation `z` and post-activation `a` in a list during `foward()`.
- You'll need them all during the backward pass.

## Backward Pass (chain rule, output → input)

### 1. Output layer gradient
- Loss = (output - target)^2
- `d_loss/d_output` = 2 * (output - target)

### 2. Pre-activation gradient (at any layer l)
- `a = sigmoid(z)` → `d_a/d_z = a * (1 - a)`
- `d_loss/d_z = d_loss/d_a * a * (1 - a)`

### 3. Weight gradient
- `z_l = W_l @ a_{l-1}`
- `d_loss/d_W_l` = `d_loss/d_z_l` ⨀ `a_{l-1}`  (outer product, shape: output_neurons × input_neurons)
- In numpy: `np.outer(delta, a_prev)`

### 4. Error propagation to previous layer
- `a[j]` appears in every `z[n]` multiplied by `W[n, j]`
- `d_loss/d_a[j]` = sum_n (`d_loss/d_z[n]` * `W[n, j]`)
- In numpy: `W.T @ delta` (matrix multiply)

### 5. Weight update
- `W_new = W_old - lr * d_loss/d_W`
- Do this for every layer from last to first.

## Known bugs
1. `_relu` uses scalar `max()` -- needs `np.maximum(0, x)` for arrays.
2. `train` returns after computing error -- no weight updates implemented yet.
3. `pandas` not installed in venv, but only used in the last cell for data display.

In [ ]:
import numpy as np
import pandas as pd
#matrixial form
# 1 matriz por layer, logo
class DenseNeuralNetwork():
    def __init__(self, layer_sizes):
        # layer_sizes = (2, 5, 2) means: input=2, hidden=5, output=2
        self.weights = []
        for i in range(len(layer_sizes) - 1):
            w = np.random.randn(layer_sizes[i+1], layer_sizes[i]) 
            self.weights.append(w)
        self.weights = np.array(self.weights, dtype=object) 

    def _relu(self,x):
        return max(0,x)
    
    def _fowardpass(self,X:np.array,n):
        return np.dot(self.weights[n],X)
    
    def _sigmoid(self,X):
        return 1/(1+np.exp(-X))
    
    def foward(self,X):
        z = X
        for i in range(self.weights.shape[0]):
            z = self._fowardpass(z,i)
            z = self._sigmoid(z)
        return z
    
    def EQM(self,x,y):
        return (x-y)**2

    def train(self,X,Y):
        error = 0
        for i in range(len(X)):
            out = self.foward(X[i])
            error += self.EQM(out,Y[i])
        error = error/i
        return error

        

        
        


In [ ]:
NN = DenseNeuralNetwork((2,4,4,3,3,2))


In [ ]:
NN.foward(np.array([3,2]))

In [ ]:
data = {"input":[],
        "output":[]}

R = 49

for i in range(10):
    for j in range(10):
        data['input'].append(np.array([i,j]))
        if i**2 + j**2 > R**2:
            data['output'].append(0)
        else:
            data['output'].append(1)

data = pd.DataFrame(data)
data